In [ ]:
#compute optimal thershold sex race

In [1]:
import os
import sys
import json
import copy
import socket
import getpass
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
sys.path.append('../')
import pickle
import warnings
from sklearn.metrics import recall_score, matthews_corrcoef, roc_auc_score, f1_score
from collections import defaultdict
import json

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

def flatten(l):
    return [item for sublist in l for item in sublist]

def compute_opt_thres(target, pred):
    opt_thres = 0
    opt_f1 = 0
    for i in np.arange(0.001, 0.999, 0.001):
        f1 = f1_score(target, pred >= i)
        if f1 >= opt_f1:
            opt_thres = i
            opt_f1 = f1
    return opt_thres

plt.rcParams.update({'font.size': 20})

In [2]:
#root_dir = Path(f'/path/to/your/root')
root_dir = Path(f'/home/lchanch/initial_training/output_sweep_12/grid_sex_race_mimic_12')

In [3]:
if Path('opt_thres.pkl').is_file():
    already_computed = set(pickle.load(Path('opt_thres.pkl').open('rb')).keys())
else:
    already_computed = set()

In [4]:
results = []
for i in tqdm(root_dir.glob('**/done')):
    args = json.load((i.parent/'args.json').open('r'))
    if (args['dataset'][0], args['task'], args['attr'], args['algorithm']) in already_computed:
        continue
    
    final_res = pickle.load((i.parent/'final_results.pkl').open('rb'))
    
    ssets = ['va', 'te', 'MIMIC-sex-te', 'CheXpert-sex-te']
  
    for sset in ssets:
        if sset in final_res:
            args[f'{sset}_auroc'] = final_res[sset]['overall']['AUROC']
            if sset == 'va':
                args[f'{sset}_min_attr_auroc'] = final_res[sset]['min_attr']['AUROC']
    args['va_y'] = final_res['va']['y']
    args['va_preds'] = final_res['va']['preds']
    
    results.append(args)
df = pd.DataFrame(results)

136it [00:00, 647.50it/s]


In [5]:
df['dataset'] = df['dataset'].apply(lambda x: x[0])

In [6]:
df.shape

(136, 33)

In [7]:
df.head()

,store_name,dataset,task,attr,group_def,algorithm,output_dir,data_dir,hparams,hparams_seed,...,checkpoint_freq,skip_model_save,debug,image_arch,aug,va_auroc,va_min_attr_auroc,te_auroc,va_y,va_preds
0,ee1b3409c45f1f63ee9c12331c50c43f,MIMIC,Cardiomegaly,sex_ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},11,...,1000,False,False,densenet_sup_in1k,basic2,0.507905,0.442790,0.502792,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...","[1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, ..."
1,df17fb5b710bd737e6c92b7b53c72bbb,MIMIC,Pneumothorax,sex_ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},3,...,1000,False,False,densenet_sup_in1k,basic2,0.658319,0.544190,0.647634,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.051433455, 0.047487833, 0.053067777, 0.0505..."
2,debb054b4300957db6ecba1946563134,MIMIC,Pleural Effusion,sex_ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},4,...,1000,False,False,densenet_sup_in1k,basic2,0.567034,0.554179,0.570472,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
3,03508c008a3edfcc4c4a65908557a259,MIMIC,No Finding,sex_ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},3,...,1000,False,False,densenet_sup_in1k,basic2,0.778254,0.762592,0.778485,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0.14332964, 0.31895646, 0.60830444, 0.5016805..."
4,6dce6c12c447938cd0f7fcbdebc8f46e,MIMIC,Cardiomegaly,sex_ethnicity,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},3,...,1000,False,False,densenet_sup_in1k,basic2,0.740255,0.698781,0.732415,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...","[0.2725993, 0.08721788, 0.21072261, 0.1291511,..."


In [ ]:
#optimal_thresholds

In [ ]:
#best models según va_min_attr_auroc

In [8]:
best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])

/tmp/ipykernel_2672863/3613221529.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])


In [9]:
best_models

store_name  \
dataset task             attr          algorithm                                     
MIMIC   Cardiomegaly     sex_ethnicity ERM        43da53fde9b217ca8b7d6ff23d5a3fa8   
        No Finding       sex_ethnicity ERM        1a924dc85b4610007ffb0473b83881b2   
        Pleural Effusion sex_ethnicity ERM        aa71e611fa4ed23a9d6e0c51f26eb581   
        Pneumothorax     sex_ethnicity ERM        5d72ae458ded4e405994d9c8666df629   

                                                 dataset              task  \
dataset task             attr          algorithm                             
MIMIC   Cardiomegaly     sex_ethnicity ERM         MIMIC      Cardiomegaly   
        No Finding       sex_ethnicity ERM         MIMIC        No Finding   
        Pleural Effusion sex_ethnicity ERM         MIMIC  Pleural Effusion   
        Pneumothorax     sex_ethnicity ERM         MIMIC      Pneumothorax   

                                                           attr group_def  \
dataset task             attr          algorithm                            
MIMIC   Cardiomegaly     sex_ethnicity ERM        sex_ethnicity     group   
        No Finding       sex_ethnicity ERM        sex_ethnicity     group   
        Pleural Effusion sex_ethnicity ERM        sex_ethnicity     group   
        Pneumothorax     sex_ethnicity ERM        sex_ethnicity     group   

                                                 algorithm  \
dataset task             attr          algorithm             
MIMIC   Cardiomegaly     sex_ethnicity ERM             ERM   
        No Finding       sex_ethnicity ERM             ERM   
        Pleural Effusion sex_ethnicity ERM             ERM   
        Pneumothorax     sex_ethnicity ERM             ERM   

                                                                                         output_dir  \
dataset task             attr          algorithm                                                      
MIMIC   Cardiomegaly     sex_ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        No Finding       sex_ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pleural Effusion sex_ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pneumothorax     sex_ethnicity ERM        /home/lchanch/initial_training/output_sweep_12...   

                                                                                 data_dir  \
dataset task             attr          algorithm                                            
MIMIC   Cardiomegaly     sex_ethnicity ERM        /home/lchanch/initial_training/image_df   
        No Finding       sex_ethnicity ERM        /home/lchanch/initial_training/image_df   
        Pleural Effusion sex_ethnicity ERM        /home/lchanch/initial_training/image_df   
        Pneumothorax     sex_ethnicity ERM        /home/lchanch/initial_training/image_df   

                                                 hparams  hparams_seed  ...  \
dataset task             attr          algorithm                        ...   
MIMIC   Cardiomegaly     sex_ethnicity ERM            {}             9  ...   
        No Finding       sex_ethnicity ERM            {}             1  ...   
        Pleural Effusion sex_ethnicity ERM            {}             1  ...   
        Pneumothorax     sex_ethnicity ERM            {}             1  ...   

                                                  checkpoint_freq  \
dataset task             attr          algorithm                    
MIMIC   Cardiomegaly     sex_ethnicity ERM                   1000   
        No Finding       sex_ethnicity ERM                   1000   
        Pleural Effusion sex_ethnicity ERM                   1000   
        Pneumothorax     sex_ethnicity ERM                   1000   

                                                 skip_model_save  debug  \
dataset task             attr          algorithm                          
MIMIC   Cardiomegaly     sex_eth

In [10]:
opt_thres = {}
for idx, row in tqdm(best_models.iterrows(), total = len(best_models)):
    dataset, task, attr, algorithm = idx
#     if dataset not in opt_thres:
#         opt_thres[dataset] = {}
    opt_thres[(dataset, task, attr, algorithm)] = np.round(compute_opt_thres(row['va_y'], row['va_preds']), 3)

100%|██████████| 4/4 [00:21<00:00,  5.33s/it]


In [11]:
if Path('opt_thres.pkl').is_file():
    old_file = pickle.load(Path('opt_thres.pkl').open('rb'))
else:
    old_file = {}

In [12]:
opt_thres = {**old_file, **opt_thres}

In [13]:
opt_thres

{('MIMIC', 'Cardiomegaly', 'sex_ethnicity', 'ERM'): 0.191,
 ('MIMIC', 'No Finding', 'sex_ethnicity', 'ERM'): 0.425,
 ('MIMIC', 'Pleural Effusion', 'sex_ethnicity', 'ERM'): 0.37,
 ('MIMIC', 'Pneumothorax', 'sex_ethnicity', 'ERM'): 0.132}

In [ ]:
#se observa que los opt thersholds son los mismos excepto para cardiomegaly, que en el de 3 semillas era 0.245

In [14]:
pickle.dump(opt_thres, open('opt_thres_sex_race_mimic_12.pkl', 'wb'))

In [15]:
best_models[['store_name','hparams_seed']]

store_name  \
dataset task             attr          algorithm                                     
MIMIC   Cardiomegaly     sex_ethnicity ERM        43da53fde9b217ca8b7d6ff23d5a3fa8   
        No Finding       sex_ethnicity ERM        1a924dc85b4610007ffb0473b83881b2   
        Pleural Effusion sex_ethnicity ERM        aa71e611fa4ed23a9d6e0c51f26eb581   
        Pneumothorax     sex_ethnicity ERM        5d72ae458ded4e405994d9c8666df629   

                                                  hparams_seed  
dataset task             attr          algorithm                
MIMIC   Cardiomegaly     sex_ethnicity ERM                   9  
        No Finding       sex_ethnicity ERM                   1  
        Pleural Effusion sex_ethnicity ERM                   1  
        Pneumothorax     sex_ethnicity ERM                   1